<a href="https://colab.research.google.com/github/sumitjhadev/RNN-for-Sentiment-Analysis/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN For Snetiment Analysis

In [8]:
import pandas as pd

In [10]:
df = pd.read_csv("IMDB Dataset.csv")

In [11]:
df.shape

(50000, 2)

In [17]:
df.drop_duplicates(inplace=True)

In [18]:
df.shape

(49582, 2)

# Pre-processing

### 1. Converting to lowercase

In [19]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [20]:
import re

# sample_text = "abc is the word, abc" # abc => xyz

# new_text = re.sub("abc", "xyz", sample_text)

### 3. Removing punctuations

In [21]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

In [22]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 4. Removing HTML

In [23]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the Stopwords

In [24]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [25]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [26]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)

In [27]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [28]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti. filming technique u...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jke) thks 's zom...,negative
4,"petter mttei's ""love time mey"" vully stun...",positive


### 6. Stemming

In [29]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [30]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [31]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,positive
1,wder ltle producti . film techniqu unssuming- ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,negative
4,petter mttei 's `` love time mey '' vulli stun...,positive


### 7. Encoding

In [32]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [33]:
y = df["sentiment"]

In [34]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


### 8. Vectorization

In [35]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,1
1,wder ltle producti . film techniqu unssuming- ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,0
4,petter mttei 's `` love time mey '' vulli stun...,1


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
X_train.shape

(39665, 5000)

In [39]:
X_test.shape

(9917, 5000)

In [40]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [41]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [42]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [43]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [44]:
import torch.nn as nn
import torch.optim as optim

In [45]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [46]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [47]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.18680572509765625
epoch = 2/10 and loss = 0.13167937099933624
epoch = 3/10 and loss = 0.2318389117717743
epoch = 4/10 and loss = 0.09212731570005417
epoch = 5/10 and loss = 0.22976651787757874
epoch = 6/10 and loss = 0.13875751197338104
epoch = 7/10 and loss = 0.20244984328746796
epoch = 8/10 and loss = 0.1974717676639557
epoch = 9/10 and loss = 0.29121991991996765
epoch = 10/10 and loss = 0.17935867607593536


In [48]:
 #evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.74165574266411
